In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from plots.utils.wandb_utils import get_wandb_stats

run_id = "rxlaij96"
summary, hist, config = get_wandb_stats(
    run_id,
    skip_cache=False,  # set to True to skip cache and fetch from W&B API
    wandb_run_cache_path=Path("/mnt/labstore/bespoke_olap/wandb_cache"),
)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/jwehrstein/.netrc.


✓ Run loaded: tpch_optim_q1-22_sonnet-4-6_bstorage_20260326_075715_8767
  State: failed
  Created: 2026-03-26T07:57:17Z
✓ Data fetched: 2210 turns, 136 columns
✓ W&B data cached to: /mnt/labstore/bespoke_olap/wandb_cache/1bc2ad2ed4e09bda64669ab82d031a58f74789d15b252cd19682b6687d118fbd.pkl


## Setup new git snapshotter folder

In [3]:
from llm_cache.git_snapshotter import GitSnapshotter

cache_repo = "git://c01/bespoke_cache.git"

# relative path to 'tmp'
workspace_path = Path.cwd() / "tmp"
workspace_path.mkdir(exist_ok=True)

# could take some time to initialize as it clones the repo and sets up git
snapshotter = GitSnapshotter(
    cache_repo=cache_repo,
    working_dir=workspace_path,
    extra_gitignore=[],
    do_not_snapshot=True,  # for safety - we don't want to accidentally commit anything from this notebook
)

## Identify snapshots to check out

In [4]:
from plots.utils.wandb_trace_preprocessor import DataCleaner

worked_on_queries = DataCleaner.extract_worked_on_queries(
    hist, drill_down_to_query_level=False
)
worked_on_spans = DataCleaner.extract_worked_on_spans(worked_on_queries)

Section 'pin & trace' starts at index: 1.
Section 'optim card' starts at index: 583.
Section 'optim trace' starts at index: 1960.
Setting rows up to index 1 to 'storage'.


In [6]:
# extract code versions for each stage
snapshot_dict = dict()


# get initial snapshot (before any stages have been worked on)
snapshot_dict["initial"] = hist["code/snapshot_hash"][0]

for i, span in enumerate(worked_on_spans):
    end_snapshot = hist["code/snapshot_hash"][
        span.end
    ]  # get snapshot hash at end of span

    if i == 0 and end_snapshot == snapshot_dict["initial"]:
        # if the snapshot hash at the end of the first span is the same as the initial snapshot, it means that the first span did not actually involve any code changes. In this case, we can skip adding this snapshot to the snapshot_dict to avoid confusion.
        continue
    snapshot_dict[span.section] = end_snapshot

print(snapshot_dict)

{'initial': '83c36d1860084c362418af81196ea0b4f5856d53', 'pin & trace': 'e74f45f69cd1299c21138c00b840696570a98cc5', 'optim card': 'c13e05ca5a47a2dff08ba80e64a7a1e8b0d14216', 'optim trace': '88b48b15d71df1c33574dd3fc11cc608600674fc'}


## Go over snapshots and extract query impl files

In [30]:
def extract_query_from_name(query_file: Path) -> str:
    name = query_file.stem  # get filename without extension
    assert name.startswith("query"), (
        f"Expected filename to start with 'query', got {name}"
    )
    qid = name[len("query") :]  # get part of filename after "query"
    return qid

In [33]:
code_dict = dict()
for stage, snapshot in snapshot_dict.items():
    # load the snapshot
    snapshotter.restore(snapshot)

    # extract all files in workspace that match the pattern "query_[0-9a-zA-Z_]+\.cpp" or .hpp

    query_files = list(snapshotter.working_dir.glob("query[0-9a-zA-Z]*.[ch]pp"))

    # extract all query names
    query_names = set([extract_query_from_name(qf) for qf in query_files])

    per_query_dict = dict()
    for query_name in sorted(query_names):
        cpp_file = snapshotter.working_dir / f"query{query_name}.cpp"
        hpp_file = snapshotter.working_dir / f"query{query_name}.hpp"
        assert cpp_file.exists()
        assert hpp_file.exists()

        # read the contents of the files
        cpp_content = cpp_file.read_text()
        hpp_content = hpp_file.read_text()
        per_query_dict[query_name] = {
            "cpp": cpp_file.read_text(),
            "hpp": hpp_file.read_text(),
        }

    per_query_dict["db_loader"] = {
        "cpp": (snapshotter.working_dir / "db_loader.cpp").read_text(),
        "hpp": (snapshotter.working_dir / "db_loader.hpp").read_text(),
    }

    code_dict[stage] = per_query_dict

In [34]:
code_dict

{'initial': {'1': {'cpp': '#include "query1.hpp"\n\n#include <algorithm>\n#include <charconv>\n#include <cstdint>\n#include <iomanip>\n#include <limits>\n#include <sstream>\n#include <stdexcept>\n#include <string>\n#include <string_view>\n#include <utility>\n#include <vector>\n\n// SELECT l_returnflag, l_linestatus,\n//   sum(l_quantity) as sum_qty, sum(l_extendedprice) as sum_base_price,\n//   sum(l_extendedprice*(1-l_discount)) as sum_disc_price,\n//   sum(l_extendedprice*(1-l_discount)*(1+l_tax)) as sum_charge,\n//   avg(l_quantity) as avg_qty, avg(l_extendedprice) as avg_price,\n//   avg(l_discount) as avg_disc, count(*) as count_order\n// FROM lineitem\n// WHERE l_shipdate <= date \'1998-12-01\' - interval \'[DELTA]\' day\n// GROUP BY l_returnflag, l_linestatus\n// ORDER BY l_returnflag, l_linestatus\n\n// ---------------------------------------------------------------------------\n// Date helpers (days since 1992-01-01 = our DATE16 encoding)\n// ----------------------------------

In [ ]:
# store as json

target_path = (
    f"/mnt/labstore/bespoke_olap/code_versions_dict/{run_id}_code_versions.json"
)

import json

with open(target_path, "w") as f:
    json.dump(code_dict, f, indent=4)

print(f"Code versions for run {run_id} stored at {target_path}")

Code versions for run rxlaij96 stored at /mnt/labstore/bespoke_olap/code_versions_dict/rxlaij96_code_versions.json
